In [1]:
# Setup and imports
import sys
from pathlib import Path

project_root = Path.cwd().parent.parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from IPython.display import display, Markdown

from ocr_vs_vlm.results_final.shared.colors import MODEL_COLORS, get_model_color, get_dataset_color
from ocr_vs_vlm.results_final.shared.stats_utils import bootstrap_ci, wilcoxon_test, cohens_d, compute_significance_matrix, format_pvalue
from ocr_vs_vlm.results_final.shared.viz_utils import setup_plotly_template, create_metric_boxplot, create_grouped_bar_chart
from ocr_vs_vlm.results_final.shared.data_loader import load_experiment_data, extract_models_from_columns, extract_metric_scores

setup_plotly_template()
print("✓ Setup complete")

✓ Setup complete


## 1. Load QA3 Experiment Data

In [2]:
QA3_PHASES = ['QA-VLM_direct_simple', 'QA-VLM_direct_detailed']
DATASETS = ['DocVQA_mini', 'InfographicVQA_mini']

all_data = []
for dataset in DATASETS:
    for phase in QA3_PHASES:
        try:
            df = load_experiment_data(dataset, phase)
            all_data.append(df)
            print(f"✓ Loaded {dataset}/{phase}: {len(df)} samples")
        except FileNotFoundError:
            print(f"✗ Missing {dataset}/{phase}")

df_qa3 = pd.concat(all_data, ignore_index=True)
models = extract_models_from_columns(df_qa3)
print(f"\n📊 Total: {len(df_qa3)} samples | Models: {models}")

✗ Missing DocVQA_mini/QA-VLM_direct_simple
✗ Missing DocVQA_mini/QA-VLM_direct_detailed
✗ Missing InfographicVQA_mini/QA-VLM_direct_simple
✗ Missing InfographicVQA_mini/QA-VLM_direct_detailed


ValueError: No objects to concatenate

## 2. VLM Model Comparison

In [ ]:
anls_scores = extract_metric_scores(df_qa3, 'anls_score')

summary_rows = []
for model, scores in anls_scores.items():
    scores_clean = scores[~np.isnan(scores)]
    mean, ci_lo, ci_hi = bootstrap_ci(scores_clean, 'mean')
    summary_rows.append({
        'Model': model,
        'ANLS Mean': mean,
        'ANLS CI (95%)': f"[{ci_lo:.4f}, {ci_hi:.4f}]",
        'N': len(scores_clean)
    })

summary_df = pd.DataFrame(summary_rows).sort_values('ANLS Mean', ascending=False)
display(summary_df)

fig = create_metric_boxplot(anls_scores, metric_name='ANLS Score', 
                            title='QA3 (Direct VQA): ANLS by VLM Model')
fig.show()

In [ ]:
# Statistical tests
if len(anls_scores) >= 2:
    p_values, significance = compute_significance_matrix(anls_scores, test='wilcoxon')
    
    print("📊 Pairwise Wilcoxon Tests (Bonferroni corrected)")
    print("-" * 60)
    for (a, b), p in p_values.items():
        d = cohens_d(anls_scores[a], anls_scores[b])
        sig = "✓" if significance[(a, b)] else "✗"
        print(f"{a} vs {b}: p={format_pvalue(p)}, d={d:.3f} ({sig})")

## 3. Prompt Effect (Simple vs Detailed)

In [ ]:
PROMPT_LABELS = {'QA-VLM_direct_simple': 'Simple', 'QA-VLM_direct_detailed': 'Detailed'}

phase_stats = []
for phase in QA3_PHASES:
    phase_df = df_qa3[df_qa3['phase'] == phase]
    for model in models:
        col = f'anls_score_{model}'
        if col in phase_df.columns:
            scores = phase_df[col].dropna().values
            if len(scores) > 0:
                mean, ci_lo, ci_hi = bootstrap_ci(scores, 'mean')
                phase_stats.append({
                    'Prompt': PROMPT_LABELS[phase],
                    'Model': model,
                    'ANLS': mean,
                    'Error': mean - ci_lo
                })

phase_df = pd.DataFrame(phase_stats)

fig = create_grouped_bar_chart(phase_df, x_col='Prompt', y_col='ANLS', group_col='Model',
                               title='QA3: Direct VQA - Prompt Effect', error_col='Error')
fig.show()

## 4. Summary

In [ ]:
print("=" * 70)
print("QA3 (DIRECT VQA) EXPERIMENT SUMMARY")
print("=" * 70)

if len(phase_df) > 0:
    best = phase_df.loc[phase_df['ANLS'].idxmax()]
    print(f"\n🏆 Best: {best['Model']} + {best['Prompt']} prompt → ANLS={best['ANLS']:.4f}")

print(f"\n📊 Model Ranking:")
for _, row in summary_df.iterrows():
    print(f"   {row['Model']}: {row['ANLS Mean']:.4f}")

print("\n" + "=" * 70)